# 📙 Bài 2 — Pandas

**Tuần 7** · Pandas là Excel dành cho lập trình viên, và là công cụ bạn dùng nhiều nhất trong công việc dữ liệu thực tế.

> Cần chạy trước: `python curriculum/03-data-toolkit/tao_du_lieu.py`

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 20)

DATA = Path("../../../data")          # từ thư mục lessons/
if not (DATA / "ban_hang.csv").exists():
    DATA = Path("data")               # nếu chạy từ thư mục gốc
print("Đọc dữ liệu từ:", DATA.resolve())

## 1. Năm lệnh đầu tiên với MỌI dataset

Nhận file lạ, **luôn** chạy đúng năm lệnh này trước khi làm bất cứ điều gì.

In [ ]:
df = pd.read_csv(DATA / "ban_hang.csv")

df.shape          # ① To cỡ nào?

In [ ]:
df.head(10)       # ② Dữ liệu trông ra sao?

In [ ]:
df.info()         # ③ Kiểu dữ liệu có đúng không? Cột nào thiếu?

> 🔍 Nhìn cột `don_gia`: kiểu `object` nghĩa là **chuỗi**, không phải số! Đó là vì trong file có những giá trị dạng `"12.000.000"`. Đây chính là loại vấn đề `info()` giúp bạn phát hiện ngay từ phút đầu.

In [ ]:
df.describe()     # ④ Có giá trị vô lý không?

In [ ]:
df.isna().mean().sort_values(ascending=False)   # ⑤ TỶ LỆ thiếu từng cột

> Dùng `.mean()` (tỷ lệ) thay vì `.sum()` (số tuyệt đối). *"500 ô thiếu"* không nói lên gì nếu bạn không biết tổng số dòng; *"6% thiếu"* thì biết ngay phải làm gì.

### Ba lệnh bổ sung nên chạy luôn

In [ ]:
print("Số dòng trùng lặp hoàn toàn:", df.duplicated().sum())
print()
print("Phân bố kênh bán:")
print(df["kenh"].value_counts())
print()
print("Số giá trị khác nhau mỗi cột:")
print(df.nunique())

**Đọc kết quả `nunique()` — bốn dấu hiệu:**

| Dấu hiệu | Nghĩa |
|---|---|
| `nunique == 1` | Cột **hằng số** — không có giá trị cho model, xoá đi |
| `nunique == số dòng` | Cột **ID** — dùng để ghép bảng, **không** đưa vào model |
| `nunique` nhỏ ở cột số | Có thể thật ra là biến hạng mục |
| `nunique` lớn bất thường ở cột hạng mục | Viết không nhất quán (`"HN"` vs `"Ha Noi"`) |

> 💡 **Câu hỏi quan trọng nhất không nằm trong code:** *"Mỗi dòng ở đây đại diện cho cái gì?"* Một đơn hàng, hay một sản phẩm trong đơn hàng? Hiểu sai điều này làm sai toàn bộ phân tích về sau.

## 2. Chọn dữ liệu — `loc` vs `iloc`

In [ ]:
print(df["danh_muc"].head(3))          # một cột -> Series
print()
print(df[["don_id", "danh_muc"]].head(3))   # nhiều cột -> DataFrame

In [ ]:
print("loc  (NHÃN, lấy CẢ cận phải):")
print(df.loc[0:2, "don_id":"danh_muc"])
print()
print("iloc (VỊ TRÍ, KHÔNG lấy cận phải):")
print(df.iloc[0:2, 0:4])

> 🔑 **`loc` = label, `iloc` = integer.** Nhớ được câu này là hết 70% bối rối với pandas.

**Vì sao `loc` lấy cả cận phải?** Vì với nhãn, `df.loc["A":"C"]` không có khái niệm "nhãn kế tiếp sau C" — phải bao gồm C mới trực giác. Với số nguyên, pandas theo quy ước slicing của Python.

In [ ]:
# ⚠️ Bẫy: sau khi lọc, index KHÔNG còn liên tục
sub = df[df["danh_muc"] == "Laptop"]
print("Index sau khi lọc:", list(sub.index[:5]))

sub = sub.reset_index(drop=True)
print("Sau reset_index   :", list(sub.index[:5]))

## 3. Lọc

In [ ]:
# Chuẩn bị: đổi don_gia sang số để lọc được
df["don_gia_so"] = pd.to_numeric(
    df["don_gia"].astype(str).str.replace(".", "", regex=False), errors="coerce"
)

dat = df[(df["don_gia_so"] > 20_000_000) & (df["kenh"] == "Website")]
print(f"Đơn đắt bán qua Website: {len(dat)} đơn")
print(dat[["don_id", "danh_muc", "don_gia_so"]].head())

In [ ]:
# .isin() thay cho chuỗi điều kiện ==
print(len(df[df["kenh"].isin(["App", "Website"])]), "đơn qua kênh online")

# .query() — cú pháp dễ đọc hơn khi điều kiện dài
print(len(df.query("don_gia_so > 20_000_000 and kenh == 'Website'")), "đơn (dùng query)")

# Lọc theo chuỗi
print(len(df[df["danh_muc"].str.contains("Laptop", na=False)]), "đơn laptop")

Nhắc lại ba quy tắc: dùng **`&` `|` `~`** chứ không phải `and` `or` `not`; **bọc ngoặc từng vế**; dùng **`.isin()`** thay cho chuỗi `==` nối bằng `|`.

## 4. Tạo cột mới

In [ ]:
d = df.copy()
d["so_luong_so"] = pd.to_numeric(d["so_luong"], errors="coerce")
d["thanh_tien"] = d["so_luong_so"] * d["don_gia_so"] * (1 - d["giam_gia"])

d["phan_khuc"] = pd.cut(
    d["thanh_tien"],
    bins=[0, 5e6, 20e6, 50e6, float("inf")],
    labels=["Nhỏ", "Vừa", "Lớn", "Rất lớn"],
)

print(d[["don_id", "thanh_tien", "phan_khuc"]].dropna().head())
print()
print(d["phan_khuc"].value_counts().sort_index())

`pd.cut()` biến biến **liên tục** thành **hạng mục** theo khoảng. Rất hay dùng để phân khúc khách hàng, nhóm tuổi, bậc giá.

## 5. `groupby` — split, apply, combine

In [ ]:
bang = (
    d.groupby("danh_muc")
     .agg(
         doanh_thu=("thanh_tien", "sum"),
         so_don=("don_id", "count"),
         gia_tri_tb=("thanh_tien", "mean"),
         gia_cao_nhat=("don_gia_so", "max"),
     )
     .reset_index()
     .sort_values("doanh_thu", ascending=False)
)
bang["doanh_thu"] = bang["doanh_thu"].round(0)
bang["gia_tri_tb"] = bang["gia_tri_tb"].round(0)
bang

**Cú pháp *named aggregation*** — `ten_moi=("cot_nguon", "ham")` — là cách hiện đại và dễ đọc nhất. Cách cũ `agg({"cot": ["sum","mean"]})` tạo ra cột đa tầng rất khó làm việc tiếp.

> ⚠️ **`count` vs `size`:** `count` **bỏ qua** ô `NaN`, `size` đếm tất cả. Nếu cột dùng để đếm có giá trị thiếu, hai hàm cho kết quả **khác nhau** — nguồn sai lệch số liệu rất âm thầm. Khi đếm số dòng, an toàn nhất là đếm trên một cột chắc chắn không bao giờ thiếu (như `id`).

In [ ]:
# Nhóm theo NHIỀU cột
d.groupby(["danh_muc", "kenh"])["thanh_tien"].sum().unstack().round(0)

### `agg` vs `transform` — khác biệt mấu chốt

In [ ]:
print("agg (giảm số dòng):")
print(d.groupby("danh_muc")["thanh_tien"].sum().round(0))
print()
print("transform (giữ nguyên số dòng):")
print(d.groupby("danh_muc")["thanh_tien"].transform("sum").head().round(0))

In [ ]:
# Nhờ transform giữ nguyên index nên chia trực tiếp được
d["ty_le_trong_danh_muc"] = (
    d["thanh_tien"] / d.groupby("danh_muc")["thanh_tien"].transform("sum")
)
print(d[["don_id", "danh_muc", "thanh_tien", "ty_le_trong_danh_muc"]].dropna().head())

**Ba ứng dụng thực tế của `transform`** — bạn sẽ dùng rất nhiều:

```python
# Tỷ lệ đóng góp trong nhóm
df["ty_le"] = df["tien"] / df.groupby("tp")["tien"].transform("sum")

# Điền thiếu bằng median CỦA NHÓM (tốt hơn median toàn cục)
df["gia"] = df["gia"].fillna(df.groupby("dm")["gia"].transform("median"))

# So với trung bình nhóm
df["cao_hon_tb"] = df["tien"] > df.groupby("tp")["tien"].transform("mean")
```

## 6. Ghép bảng — và cái bẫy chết người

In [ ]:
khach = pd.read_csv(DATA / "khach_hang.csv")
print("Đơn hàng:", d.shape, " Khách hàng:", khach.shape)

truoc = len(d)
gop = pd.merge(d, khach, on="khach_id", how="left")
print(f"\nSau merge: {truoc} -> {len(gop)} dòng")
print("Số đơn không tìm thấy khách:", gop["ho_ten"].isna().sum())

In [ ]:
# Công cụ 1: validate — pandas tự báo lỗi nếu khoá bên phải bị trùng
gop = pd.merge(d, khach, on="khach_id", how="left", validate="many_to_one")

# Công cụ 2: indicator — xem dòng nào khớp, dòng nào không
kiem_tra = pd.merge(d, khach, on="khach_id", how="left", indicator=True)
print(kiem_tra["_merge"].value_counts())

In [ ]:
# 🔬 Xem chuyện gì xảy ra khi khoá bên phải BỊ TRÙNG
khach_hong = pd.concat([khach, khach.head(3)])      # nhân đôi 3 khách
thu = pd.merge(d, khach_hong, on="khach_id", how="left")
print(f"Số dòng: {len(d)} -> {len(thu)}   (+{len(thu) - len(d)} dòng ma!)")
print("\nDoanh thu bị thổi phồng:")
print(f"  Thật  : {d['thanh_tien'].sum():,.0f}")
print(f"  Sau khi merge hỏng: {thu['thanh_tien'].sum():,.0f}")

> ⚠️ **Đây là bug âm thầm nguy hiểm nhất khi xử lý dữ liệu.** Không có thông báo lỗi nào. Bạn chỉ phát hiện khi sếp hỏi *"sao tháng này doanh thu tăng 40%?"*
>
> **Luôn kiểm tra `.shape` trước và sau mọi lần merge.**

| `how` | Tác dụng |
|---|---|
| `"left"` | Giữ hết dòng bên trái — **an toàn nhất, mặc định nên dùng** |
| `"inner"` | Chỉ giữ dòng khớp cả hai — âm thầm làm mất dữ liệu |
| `"outer"` | Giữ hết cả hai bên |
| `"right"` | Giữ hết bên phải |

Người mới hay dùng `inner` theo quán tính rồi mất 30% dữ liệu mà không biết.

## 7. Chuỗi & thời gian

In [ ]:
print(khach["ho_ten"].head(8).tolist())
print()
print("Sau khi chuẩn hoá:")
print(khach["ho_ten"].str.strip().str.title().head(8).tolist())

In [ ]:
d["ngay"] = pd.to_datetime(d["ngay_dat"], format="%Y-%m-%d", errors="coerce")
print("Đổi được:", d["ngay"].notna().sum(), "/", len(d), "dòng")
print("(phần còn lại có định dạng khác — sẽ xử lý ở Tuần 8)")
print()

hop_le = d.dropna(subset=["ngay"])
print("Doanh thu theo tháng:")
print(hop_le.set_index("ngay").resample("ME")["thanh_tien"].sum().round(0))

Sau khi cột đã là kiểu `datetime`, cả kho báu mở ra: `.dt.month`, `.dt.year`, `.dt.dayofweek`, `.dt.quarter`, `resample()`…

Chúng **chỉ hoạt động khi cột đã là datetime**. Nếu vẫn là chuỗi thì `.dt` báo lỗi — đó là lý do đổi kiểu ngày là một trong những việc đầu tiên phải làm.

## ✍️ Bài tập

```bash
pytest tests/phase03/test_ex02.py -v
```